# Confound test: is it route position?

Section 6.2 claims the bespoke anchors worked because their targets are
mostly off the optimal route, which removes route position as a usable
cue and leaves the period-B self-loop as the only one. That is currently
an inference from two generators that differ in several ways. This tests
it directly.

| anchor set | targets on the optimal route | localize |
| --- | --- | --- |
| bespoke `anchors.py` | 4/20 (20%) | **32/32** |
| `anchors_v22.py`, `silent_break` only | 20/20 (100%) | 7/32 |
| **this run** | **4/20 (20%)** | ? |

The manipulation uses `irrelevant`, which under v2.2 sets the target link
to p = 0 in both modes exactly as `silent_break` does. So the surface
signature is identical, the target is dead in period B in all 20 changed
worlds either way, and the only thing that moves is where the target sits
relative to the optimal route. Cycling four `irrelevant` worlds per
`silent_break` reproduces the bespoke generator's 20% on-route rate.

Everything else is held at the `augustopt` settings: 40 worlds, 140
examples, deterministic only, `PRES_REPEAT = 1`, `GRAD_ACCUM = 4`, 105
optimizer steps.

## What each outcome means

* **Near 32/32** -> route position is the confound. The mechanism
  paragraph in 6.2 is right, and the good generator can be fixed by
  mixing conditions rather than reverting to the bespoke one.
* **Near 7/32** -> route position is not the explanation. Something else
  distinguishes the two generators, and that paragraph has to come out of
  the draft.

Either way the claim stops being an inference.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate

## Preflight

In [ ]:
import sys, glob, json, time, collections
import torch

def find_dir(marker, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

RUN_TAG   = "offroute"
REPO_PATH = find_dir("resource_mdp.py")
EVAL_PATH = find_dir("anchors_v22.py")
GEN_PATH  = find_dir("gen_payloads.py")   # may sit in a different folder
OUT_DIR   = "/kaggle/working"
print("repo:        ", REPO_PATH)
print("eval:        ", EVAL_PATH)
print("gen_payloads:", GEN_PATH)

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
assert torch.cuda.device_count() == 1, (
    "CUDA_VISIBLE_DEVICES did not take; restart the kernel and run from the "
    "first cell")
CAP = torch.cuda.get_device_capability(0)
USE_BF16 = CAP[0] >= 8          # T4 is 7.5 and only emulates bf16
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"gpu: {torch.cuda.get_device_name(0)} | dtype: {DTYPE}")

import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_OUT = f"{OUT_DIR}/anchor_adapter_{RUN_TAG}_{MODEL_NAME.split('/')[-1]}"

# four irrelevant worlds per silent_break world gives 20% on-route,
# matching the bespoke generator
MIX         = ("irrelevant",) * 4 + ("silent_break",)
N_WORLDS    = 40
K           = 5
TOTAL_STEPS = 105
LR          = 1e-4
GRAD_ACCUM  = 4
MAX_LEN     = 2048
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.0
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
           "gate_proj", "up_proj", "down_proj"]

sys.path.insert(0, EVAL_PATH)
sys.path.insert(0, GEN_PATH)
import anchors_v22, gen_payloads as GP, ecpm_eval as E
rp = anchors_v22.load_env(REPO_PATH)
E.attach(REPO_PATH)
anchors_v22.CHANGED_CONDITIONS = MIX
print("condition cycle:", MIX)

## Build the anchors, and measure the mix

The assert is the point of the run: if the on-route rate does not land
near the bespoke generator's 20%, the manipulation did not happen and the
comparison is meaningless.

In [ ]:
worlds = anchors_v22.build_anchor_set(
    rp, n_worlds=N_WORLDS, k=K, stochastic_share=0.0,
    first_seed=1000, preservation_changed_repeat=1)
examples = anchors_v22.to_examples(worlds)
summary = anchors_v22.summarise(worlds)
for key, val in summary.items():
    print(f"  {key}: {val}")

on = off = dead = 0
for w in worlds:
    if not w["changed"]:
        continue
    sc = anchors_v22._scenario(w["seed"], w["condition"], K)
    rec = rp.build_record(sc, True)
    if rec["change"].get("on_optimal_route"):
        on += 1
    else:
        off += 1
    dead += rec["change"].get("new_p") == 0.0

print(f"\ntargets on the optimal route: {on}/{on+off} ({on/(on+off):.0%})")
print(f"targets dead in period B:     {dead}/{on+off}")
print("bespoke anchors.py: 4/20 (20%) on-route, 20/20 dead")
print("anchors_v22 silent_break only: 20/20 (100%) on-route, 20/20 dead")

assert len(examples) == 140, f"expected 140 examples, got {len(examples)}"
assert 0.10 <= on / (on + off) <= 0.35, (
    f"on-route rate is {on/(on+off):.0%}, not near the bespoke 20%; "
    "the manipulation did not take")
assert dead == on + off, "not every target is dead in period B"
json.dump({"mix": list(MIX), "on_route": on, "off_route": off,
           "summary": summary},
          open(f"{OUT_DIR}/anchor_mix_{RUN_TAG}.json", "w"), indent=1)

## Tokenize

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def as_ids(x):
    # apply_chat_template may return a BatchEncoding, which subclasses
    # UserDict rather than dict, so list(x) would give the keys
    if hasattr(x, "input_ids"):
        x = x.input_ids
    elif hasattr(x, "keys") and "input_ids" in x.keys():
        x = x["input_ids"]
    x = list(x)
    if x and isinstance(x[0], (list, tuple)):
        x = list(x[0])
    if not all(isinstance(t, int) for t in x):
        raise TypeError(f"expected token ids, got {type(x[0]).__name__}")
    return x

def encode(ex):
    pre = as_ids(tok.apply_chat_template(
        [{"role": "system", "content": E.SYSTEM},
         {"role": "user", "content": ex["prompt"]}],
        add_generation_prompt=True, tokenize=True))
    ans = as_ids(tok(ex["gold"] + tok.eos_token,
                     add_special_tokens=False)["input_ids"])
    return {"input_ids": pre + ans, "labels": [-100] * len(pre) + ans,
            "n_prefix": len(pre)}

enc = [encode(e) for e in examples]
lengths = sorted(len(x["input_ids"]) for x in enc)
print(f"{len(enc)} examples | tokens max {lengths[-1]} | "
      f"prefix min {min(x['n_prefix'] for x in enc)}")
assert lengths[-1] <= MAX_LEN
assert min(x["n_prefix"] for x in enc) > 300, "chat template did not tokenize"
assert all(any(l != -100 for l in x["labels"]) for x in enc)

from torch.utils.data import Dataset

class Anchors(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {"input_ids": r["input_ids"], "labels": r["labels"]}

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        gap = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * gap)
        out["labels"].append(b["labels"] + [-100] * gap)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * gap)
    return {k: torch.tensor(v) for k, v in out.items()}

train_ds = Anchors(enc)

## Train

In [ ]:
from transformers import (AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=TARGETS))
model.print_trainable_parameters()

result = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=f"{OUT_DIR}/offroute_tmp",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_steps=TOTAL_STEPS, learning_rate=LR,
        warmup_steps=max(1, TOTAL_STEPS // 20),
        lr_scheduler_type="cosine", logging_steps=10,
        save_strategy="no", report_to=[],
        bf16=USE_BF16, fp16=not USE_BF16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}),
    train_dataset=train_ds, data_collator=collate).train()
print(result.metrics)
print("\nfinal loss elsewhere: bespoke 0.041 | augustopt see its provenance")

model.save_pretrained(ADAPTER_OUT)
tok.save_pretrained(ADAPTER_OUT)
json.dump({"model": MODEL_NAME, "phase": 1, "run": "off_route_confound",
           "mix": list(MIX), "on_route": on, "off_route": off,
           "anchor_worlds": len(worlds), "anchor_examples": len(examples),
           "optimizer_steps": TOTAL_STEPS, "lr": LR, "grad_accum": GRAD_ACCUM,
           "lora": {"r": LORA_R, "alpha": LORA_ALPHA,
                    "dropout": LORA_DROPOUT, "targets": TARGETS},
           "final_loss": result.metrics.get("train_loss")},
          open(f"{ADAPTER_OUT}/phase1_provenance.json", "w"), indent=1)
print("saved to", ADAPTER_OUT)

## Evaluate on deterministic `silent_break`

The same 32 graded instances the other adapters were scored on, built
here rather than read from a payload folder. Localization only.

In [ ]:
import pandas as pd

rp_gp, _ = GP.load_env(REPO_PATH)
payloads, skipped = [], []
for seed in range(80):
    if len(payloads) >= 32:
        break
    try:
        payloads.append(GP.build_for_seed(rp_gp, seed, True, K, "silent_break"))
    except (ValueError, RuntimeError):
        skipped.append(seed)
print(f"{len(payloads)} graded instances: {[p['seed'] for p in payloads]}")

model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

def generate(messages, max_new_tokens):
    e = tok.apply_chat_template(messages, add_generation_prompt=True,
                                return_tensors="pt", return_dict=True)
    e = {k: v.to(model.device) for k, v in e.items()}
    with torch.no_grad():
        o = model.generate(**e, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0, e["input_ids"].shape[1]:], skip_special_tokens=True)

t0 = time.time()
rows = E.run_arm(payloads, generate, arm="arm_c", mode="single",
                 probes=["localization"],
                 out_path=f"{OUT_DIR}/raw_{RUN_TAG}_arm_c.jsonl")
ok = [r["seed"] for r in rows if r["scored"].get("correct")]
node = sum(1 for r in rows if r["parsed"].get("node") == r["target"][0])
print(f"\nlocalization {len(ok)}/{len(rows)}  node-level {node}/{len(rows)}  "
      f"({(time.time()-t0)/60:.1f} min)")
print("correct on:", sorted(ok))

In [ ]:
print(f"{'anchor set':34} {'on-route':>10} {'localize':>10}")
for name, rate, loc in (
        ("bespoke anchors.py", "4/20", "32/32"),
        ("anchors_v22, silent_break only", "20/20", "7/32"),
        ("anchors_v22, mixed (this run)", f"{on}/{on+off}",
         f"{len(ok)}/{len(rows)}"),
        ("untrained, evidence in prompt", "-", "4/32"),
        ("evidence-only rule", "-", "32/32")):
    print(f"  {name:32} {rate:>10} {loc:>10}")

df = pd.DataFrame([{"seed": r["seed"], "gold": " ".join(r["target"]),
                    "said": (f"{r['parsed'].get('node')} "
                             f"{r['parsed'].get('action')}"
                             if r["parsed"]["status"] == "ok"
                             else r["parsed"]["status"]),
                    "correct": bool(r["scored"].get("correct"))}
                   for r in sorted(rows, key=lambda r: r["seed"])])
display(df)

## Package

In [ ]:
import zipfile, shutil
shutil.rmtree(f"{OUT_DIR}/offroute_tmp", ignore_errors=True)
ZIP_PATH = f"{OUT_DIR}/{RUN_TAG}_all.zip"
n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if (not os.path.isfile(path) or os.path.basename(ZIP_PATH) in path
                or ".ipynb_checkpoints" in path):
            continue
        z.write(path, os.path.relpath(path, OUT_DIR))
        n += 1
print(f"{n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")